# Impacto Ambiental de la Energia

Este notebook construye una base de datos para el tema **Impacto ambiental de la energia** usando fuentes oficiales y estables.

Fuentes usadas en esta version:
- EIA: generacion electrica anual por estado
- EIA: emisiones anuales de CO2 por estado y fuente
- CNE Chile: proyectos PMGD, proyectos de generacion y sistemas BESS en construccion

Resultado esperado:
- mas de 100 registros
- al menos 8 etiquetas por registro
- exportacion a CSV y JSON
- carga a MongoDB para usarlo despues con Spark


In [1]:
from pathlib import Path
from datetime import datetime

import pandas as pd
from IPython.display import display
from pymongo import MongoClient

NOMBRE_GRUPO = "energia-y-sustentabilidad-1"
TEMA_PROYECTO = "Impacto ambiental de la energia"
FECHA_EXTRACCION = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

EIA_GENERATION_URL = "https://www.eia.gov/electricity/data/state/annual_generation_state.xls"
EIA_EMISSIONS_URL = "https://www.eia.gov/electricity/data/state/emission_annual.xlsx"
CNE_PROJECTS_URL = "https://www.cne.cl/wp-content/uploads/2026/03/Tablas-Declaracion-Construccion-Marzo-2026.xlsx"

PROJECT_DIR = Path("/home/jovyan/work") if Path("/home/jovyan/work").exists() else Path.cwd()
OUTPUT_DIR = PROJECT_DIR / "semanas" / "Semana 6 Integracion" / "salidas"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_SALIDA = OUTPUT_DIR / "impacto_ambiental_energia.csv"
JSON_SALIDA = OUTPUT_DIR / "impacto_ambiental_energia.json"

client = MongoClient("database", 27017)
client.admin.command("ping")
db = client["proyecto_bigdata"]
coleccion = db["impacto_ambiental_energia"]

print("Tema:", TEMA_PROYECTO)
print("Grupo:", NOMBRE_GRUPO)
print("MongoDB: conexion OK")
print("Salida CSV:", CSV_SALIDA)
print("Salida JSON:", JSON_SALIDA)


Tema: Impacto ambiental de la energia
Grupo: energia-y-sustentabilidad-1
MongoDB: conexion OK
Salida CSV: /home/jovyan/work/semanas/Semana 6 Integracion/salidas/impacto_ambiental_energia.csv
Salida JSON: /home/jovyan/work/semanas/Semana 6 Integracion/salidas/impacto_ambiental_energia.json


In [2]:
RENEWABLE_KEYWORDS = ["solar", "wind", "hydro", "geothermal", "wood", "biomass", "waste", "renewable", "fotovoltaico", "eolico", "hidro"]
FOSSIL_KEYWORDS = ["coal", "petroleum", "natural gas", "other gases", "diesel", "carbon"]
NUCLEAR_KEYWORDS = ["nuclear"]
STORAGE_KEYWORDS = ["bess", "storage", "almacenamiento"]


def clasificar_categoria(texto):
    texto = str(texto).strip().lower()
    if any(keyword in texto for keyword in STORAGE_KEYWORDS):
        return "Almacenamiento"
    if any(keyword in texto for keyword in RENEWABLE_KEYWORDS):
        return "Renovable"
    if any(keyword in texto for keyword in FOSSIL_KEYWORDS):
        return "Fosil"
    if any(keyword in texto for keyword in NUCLEAR_KEYWORDS):
        return "Nuclear"
    return "Otra"


def snake(value):
    value = str(value).strip().lower()
    replacements = {
        " ": "_",
        "\n": "_",
        ".": "",
        ",": "",
        "-": "_",
        "/": "_",
        "á": "a",
        "é": "e",
        "í": "i",
        "ó": "o",
        "ú": "u",
        "ñ": "n",
        "[": "",
        "]": "",
        "(": "",
        ")": ""
    }
    for old, new in replacements.items():
        value = value.replace(old, new)
    while "__" in value:
        value = value.replace("__", "_")
    return value.strip("_")


def serie(df, col, default=""):
    if col in df.columns:
        return df[col]
    return pd.Series([default] * len(df), index=df.index)


In [3]:
def cargar_eia_generacion():
    df = pd.read_excel(EIA_GENERATION_URL, header=1)
    df.columns = [snake(col) for col in df.columns]
    df = df.rename(columns={
        "type_of_producer": "actor",
        "energy_source": "tecnologia",
        "generation_megawatthours": "valor",
        "year": "periodo",
        "state": "region"
    })
    df = df[(df["periodo"] >= 2020) & df["region"].notna() & df["tecnologia"].notna()]
    df = df[~df["tecnologia"].isin(["Total", "All Sources"])]
    df["valor"] = pd.to_numeric(df["valor"], errors="coerce")
    df = df.dropna(subset=["valor"])
    df = df[df["valor"] > 0]

    df["fuente_sitio"] = "EIA"
    df["dataset"] = "annual_generation_state"
    df["url_origen"] = EIA_GENERATION_URL
    df["grupo"] = NOMBRE_GRUPO
    df["tema"] = TEMA_PROYECTO
    df["fecha_extraccion"] = FECHA_EXTRACCION
    df["pais"] = "Estados Unidos"
    df["indicador"] = "generacion_electrica"
    df["categoria_energia"] = df["tecnologia"].map(clasificar_categoria)
    df["item"] = df["region"].astype(str) + " - " + df["tecnologia"].astype(str)
    df["unidad"] = "MWh"

    columnas = [
        "fuente_sitio", "dataset", "url_origen", "grupo", "tema", "fecha_extraccion",
        "pais", "region", "periodo", "indicador", "categoria_energia", "tecnologia",
        "actor", "item", "valor", "unidad"
    ]
    return df[columnas]


df_eia_generacion = cargar_eia_generacion()
print("Filas EIA generacion:", len(df_eia_generacion))
display(df_eia_generacion.head(5))


Filas EIA generacion: 8013


,fuente_sitio,dataset,url_origen,grupo,tema,fecha_extraccion,pais,region,periodo,indicador,categoria_energia,tecnologia,actor,item,valor,unidad
53758,EIA,annual_generation_state,https://www.eia.gov/electricity/data/state/ann...,energia-y-sustentabilidad-1,Impacto ambiental de la energia,2026-04-22 23:59:30,Estados Unidos,AK,2020,generacion_electrica,Fosil,Coal,Total Electric Power Industry,AK - Coal,721789.0,MWh
53759,EIA,annual_generation_state,https://www.eia.gov/electricity/data/state/ann...,energia-y-sustentabilidad-1,Impacto ambiental de la energia,2026-04-22 23:59:30,Estados Unidos,AK,2020,generacion_electrica,Renovable,Hydroelectric Conventional,Total Electric Power Industry,AK - Hydroelectric Conventional,1763947.0,MWh
53760,EIA,annual_generation_state,https://www.eia.gov/electricity/data/state/ann...,energia-y-sustentabilidad-1,Impacto ambiental de la energia,2026-04-22 23:59:30,Estados Unidos,AK,2020,generacion_electrica,Fosil,Natural Gas,Total Electric Power Industry,AK - Natural Gas,2640760.0,MWh
53762,EIA,annual_generation_state,https://www.eia.gov/electricity/data/state/ann...,energia-y-sustentabilidad-1,Impacto ambiental de la energia,2026-04-22 23:59:30,Estados Unidos,AK,2020,generacion_electrica,Fosil,Petroleum,Total Electric Power Industry,AK - Petroleum,985802.0,MWh
53763,EIA,annual_generation_state,https://www.eia.gov/electricity/data/state/ann...,energia-y-sustentabilidad-1,Impacto ambiental de la energia,2026-04-22 23:59:30,Estados Unidos,AK,2020,generacion_electrica,Renovable,Other Biomass,Total Electric Power Industry,AK - Other Biomass,39065.0,MWh


In [4]:
def cargar_eia_emisiones():
    df = pd.read_excel(EIA_EMISSIONS_URL)
    df.columns = [snake(col) for col in df.columns]
    df = df.rename(columns={
        "producer_type": "actor",
        "energy_source": "tecnologia",
        "co2_metric_tons": "valor",
        "year": "periodo",
        "state": "region"
    })
    df = df[(df["periodo"] >= 2020) & df["region"].notna() & df["tecnologia"].notna()]
    df = df[~df["tecnologia"].isin(["Total", "All Sources"])]
    df["valor"] = pd.to_numeric(df["valor"], errors="coerce")
    df = df.dropna(subset=["valor"])
    df = df[df["valor"] >= 0]

    df["fuente_sitio"] = "EIA"
    df["dataset"] = "emission_annual"
    df["url_origen"] = EIA_EMISSIONS_URL
    df["grupo"] = NOMBRE_GRUPO
    df["tema"] = TEMA_PROYECTO
    df["fecha_extraccion"] = FECHA_EXTRACCION
    df["pais"] = "Estados Unidos"
    df["indicador"] = "emisiones_co2"
    df["categoria_energia"] = df["tecnologia"].map(clasificar_categoria)
    df["item"] = df["region"].astype(str) + " - " + df["tecnologia"].astype(str)
    df["unidad"] = "metric_tons_co2"

    columnas = [
        "fuente_sitio", "dataset", "url_origen", "grupo", "tema", "fecha_extraccion",
        "pais", "region", "periodo", "indicador", "categoria_energia", "tecnologia",
        "actor", "item", "valor", "unidad"
    ]
    return df[columnas]


df_eia_emisiones = cargar_eia_emisiones()
print("Filas EIA emisiones:", len(df_eia_emisiones))
display(df_eia_emisiones.head(5))


Filas EIA emisiones: 5622


,fuente_sitio,dataset,url_origen,grupo,tema,fecha_extraccion,pais,region,periodo,indicador,categoria_energia,tecnologia,actor,item,valor,unidad
43272,EIA,emission_annual,https://www.eia.gov/electricity/data/state/emi...,energia-y-sustentabilidad-1,Impacto ambiental de la energia,2026-04-22 23:59:30,Estados Unidos,AK,2020,emisiones_co2,Fosil,Coal,Electric Utility,AK - Coal,781365,metric_tons_co2
43273,EIA,emission_annual,https://www.eia.gov/electricity/data/state/emi...,energia-y-sustentabilidad-1,Impacto ambiental de la energia,2026-04-22 23:59:30,Estados Unidos,AK,2020,emisiones_co2,Fosil,Natural Gas,Electric Utility,AK - Natural Gas,1212075,metric_tons_co2
43274,EIA,emission_annual,https://www.eia.gov/electricity/data/state/emi...,energia-y-sustentabilidad-1,Impacto ambiental de la energia,2026-04-22 23:59:30,Estados Unidos,AK,2020,emisiones_co2,Fosil,Petroleum,Electric Utility,AK - Petroleum,668705,metric_tons_co2
43276,EIA,emission_annual,https://www.eia.gov/electricity/data/state/emi...,energia-y-sustentabilidad-1,Impacto ambiental de la energia,2026-04-22 23:59:30,Estados Unidos,AK,2020,emisiones_co2,Fosil,Petroleum,IPP NAICS-22 Non-Cogen,AK - Petroleum,1886,metric_tons_co2
43278,EIA,emission_annual,https://www.eia.gov/electricity/data/state/emi...,energia-y-sustentabilidad-1,Impacto ambiental de la energia,2026-04-22 23:59:30,Estados Unidos,AK,2020,emisiones_co2,Fosil,Coal,IPP NAICS-22 Cogen,AK - Coal,322269,metric_tons_co2


In [5]:
def cargar_cne_proyectos():
    frames = []
    workbook = pd.ExcelFile(CNE_PROJECTS_URL)
    hojas_objetivo = [workbook.sheet_names[2], workbook.sheet_names[3], workbook.sheet_names[4]]

    for sheet in hojas_objetivo:
        df = pd.read_excel(CNE_PROJECTS_URL, sheet_name=sheet, skiprows=2)
        df.columns = [snake(col) for col in df.columns]
        if "proyecto" not in df.columns:
            continue

        fecha = pd.to_datetime(serie(df, "fecha_estimada_de_interconexion", None), errors="coerce")
        tecnologia = serie(df, "tipo_de_tecnologia")

        base = pd.DataFrame({
            "fuente_sitio": "CNE",
            "dataset": snake(sheet),
            "url_origen": CNE_PROJECTS_URL,
            "grupo": NOMBRE_GRUPO,
            "tema": TEMA_PROYECTO,
            "fecha_extraccion": FECHA_EXTRACCION,
            "pais": "Chile",
            "region": serie(df, "ubicacion"),
            "periodo": fecha.dt.year,
            "indicador": "potencia_neta",
            "categoria_energia": tecnologia.map(clasificar_categoria),
            "tecnologia": tecnologia,
            "actor": serie(df, "propietario"),
            "item": serie(df, "proyecto"),
            "valor": pd.to_numeric(serie(df, "potencia_neta_mw"), errors="coerce"),
            "unidad": "MW"
        })
        frames.append(base)

        if "capacidad_de_almacenamiento_mwh" in df.columns:
            almacenamiento = pd.DataFrame({
                "fuente_sitio": "CNE",
                "dataset": snake(sheet),
                "url_origen": CNE_PROJECTS_URL,
                "grupo": NOMBRE_GRUPO,
                "tema": TEMA_PROYECTO,
                "fecha_extraccion": FECHA_EXTRACCION,
                "pais": "Chile",
                "region": serie(df, "ubicacion"),
                "periodo": fecha.dt.year,
                "indicador": "capacidad_almacenamiento",
                "categoria_energia": tecnologia.map(clasificar_categoria),
                "tecnologia": tecnologia,
                "actor": serie(df, "propietario"),
                "item": serie(df, "proyecto"),
                "valor": pd.to_numeric(serie(df, "capacidad_de_almacenamiento_mwh"), errors="coerce"),
                "unidad": "MWh"
            })
            frames.append(almacenamiento)

    df_final = pd.concat(frames, ignore_index=True)
    df_final = df_final.dropna(subset=["item", "valor"])
    df_final = df_final[df_final["valor"] > 0]
    return df_final


df_cne = cargar_cne_proyectos()
print("Filas CNE:", len(df_cne))
display(df_cne.head(5))


Filas CNE: 356


,fuente_sitio,dataset,url_origen,grupo,tema,fecha_extraccion,pais,region,periodo,indicador,categoria_energia,tecnologia,actor,item,valor,unidad
0,CNE,pmgd,https://www.cne.cl/wp-content/uploads/2026/03/...,energia-y-sustentabilidad-1,Impacto ambiental de la energia,2026-04-22 23:59:30,Chile,Región de Los Lagos,2021.0,potencia_neta,Otra,PMGD Eólico,El Cruce SpA,PE El Cruce,2.9,MW
1,CNE,pmgd,https://www.cne.cl/wp-content/uploads/2026/03/...,energia-y-sustentabilidad-1,Impacto ambiental de la energia,2026-04-22 23:59:30,Chile,Región de Los Lagos,2021.0,potencia_neta,Otra,PMGD Eólico,OCHS SpA,PE OCHS,2.9,MW
2,CNE,pmgd,https://www.cne.cl/wp-content/uploads/2026/03/...,energia-y-sustentabilidad-1,Impacto ambiental de la energia,2026-04-22 23:59:30,Chile,Región Metropolitana de Santiago,2021.0,potencia_neta,Renovable,PMGD Fotovoltaico,Solarity SpA,PMGD Techos Solares Watts,0.9,MW
3,CNE,pmgd,https://www.cne.cl/wp-content/uploads/2026/03/...,energia-y-sustentabilidad-1,Impacto ambiental de la energia,2026-04-22 23:59:30,Chile,Región de La Araucanía,2021.0,potencia_neta,Renovable,PMGD Fotovoltaico,Libertador Solar 7 SpA,PMGD FV Cancura II Solar,2.8,MW
4,CNE,pmgd,https://www.cne.cl/wp-content/uploads/2026/03/...,energia-y-sustentabilidad-1,Impacto ambiental de la energia,2026-04-22 23:59:30,Chile,Región de La Araucanía,2022.0,potencia_neta,Renovable,PMGD Fotovoltaico,Libertador Solar 4 SpA,PMGD FV Nanco,2.8,MW


In [6]:
df_impacto = pd.concat([df_eia_generacion, df_eia_emisiones, df_cne], ignore_index=True)
df_impacto["periodo"] = pd.to_numeric(df_impacto["periodo"], errors="coerce").astype("Int64")
df_impacto = df_impacto.sort_values(["pais", "indicador", "region", "periodo", "item"], na_position="last").reset_index(drop=True)

print("Total de registros:", len(df_impacto))
print("Etiquetas disponibles:", len(df_impacto.columns))
print(df_impacto.columns.tolist())

assert len(df_impacto) >= 100, "La base debe tener al menos 100 registros"
assert len(df_impacto.columns) >= 8, "La base debe tener al menos 8 etiquetas"

display(df_impacto.head(10))
display(
    df_impacto.groupby(["fuente_sitio", "indicador"]).size().reset_index(name="registros").sort_values("registros", ascending=False)
)


Total de registros: 13991
Etiquetas disponibles: 16
['fuente_sitio', 'dataset', 'url_origen', 'grupo', 'tema', 'fecha_extraccion', 'pais', 'region', 'periodo', 'indicador', 'categoria_energia', 'tecnologia', 'actor', 'item', 'valor', 'unidad']


,fuente_sitio,dataset,url_origen,grupo,tema,fecha_extraccion,pais,region,periodo,indicador,categoria_energia,tecnologia,actor,item,valor,unidad
0,CNE,bess,https://www.cne.cl/wp-content/uploads/2026/03/...,energia-y-sustentabilidad-1,Impacto ambiental de la energia,2026-04-22 23:59:30,Chile,Región Del Libertador Gral. Bernardo O’Higgins,2026,capacidad_almacenamiento,Almacenamiento,BESS,Statkraft Eólico S.A.,BESS Cardonal,104.33,MWh
1,CNE,bess,https://www.cne.cl/wp-content/uploads/2026/03/...,energia-y-sustentabilidad-1,Impacto ambiental de la energia,2026-04-22 23:59:30,Chile,Región Metropolitana de Santiago,2025,capacidad_almacenamiento,Almacenamiento,BESS,AES Andes S.A.,Aumento de Inyección y Retiro desde la Red VR1...,250.00,MWh
2,CNE,bess,https://www.cne.cl/wp-content/uploads/2026/03/...,energia-y-sustentabilidad-1,Impacto ambiental de la energia,2026-04-22 23:59:30,Chile,Región Metropolitana de Santiago,2025,capacidad_almacenamiento,Almacenamiento,BESS,Consorcio Santa Marta,BESS Santa Marta,50.00,MWh
3,CNE,bess,https://www.cne.cl/wp-content/uploads/2026/03/...,energia-y-sustentabilidad-1,Impacto ambiental de la energia,2026-04-22 23:59:30,Chile,Región Metropolitana de Santiago,2026,capacidad_almacenamiento,Almacenamiento,BESS,Solar TI Veintidos SpA,BESS Chicha Solar,45.00,MWh
4,CNE,bess,https://www.cne.cl/wp-content/uploads/2026/03/...,energia-y-sustentabilidad-1,Impacto ambiental de la energia,2026-04-22 23:59:30,Chile,Región Metropolitana de Santiago,2026,capacidad_almacenamiento,Almacenamiento,BESS,Engie Energía Chile S.A.,BESS Libélula,960.10,MWh
5,CNE,bess,https://www.cne.cl/wp-content/uploads/2026/03/...,energia-y-sustentabilidad-1,Impacto ambiental de la energia,2026-04-22 23:59:30,Chile,Región Metropolitana de Santiago,2026,capacidad_almacenamiento,Almacenamiento,BESS,Tedlar Marte SpA,Lota II,11.70,MWh
6,CNE,bess,https://www.cne.cl/wp-content/uploads/2026/03/...,energia-y-sustentabilidad-1,Impacto ambiental de la energia,2026-04-22 23:59:30,Chile,Región Metropolitana de Santiago,2026,capacidad_almacenamiento,Almacenamiento,BESS,PSF Las Violetas SpA,PSF BESS Las Violetas,22.00,MWh
7,CNE,bess,https://www.cne.cl/wp-content/uploads/2026/03/...,energia-y-sustentabilidad-1,Impacto ambiental de la energia,2026-04-22 23:59:30,Chile,Región Metropolitana de Santiago,2027,capacidad_almacenamiento,Almacenamiento,BESS,Solar TI Cincuenta y Siete SpA,BESS Amuleto,36.00,MWh
8,CNE,bess,https://www.cne.cl/wp-content/uploads/2026/03/...,energia-y-sustentabilidad-1,Impacto ambiental de la energia,2026-04-22 23:59:30,Chile,Región Metropolitana de Santiago,2027,capacidad_almacenamiento,Almacenamiento,BESS,CVE Proyecto Veintidós SpA.,Isidora Solar,40.00,MWh
9,CNE,bess,https://www.cne.cl/wp-content/uploads/2026/03/...,energia-y-sustentabilidad-1,Impacto ambiental de la energia,2026-04-22 23:59:30,Chile,Región Metropolitana de Santiago,2027,capacidad_almacenamiento,Almacenamiento,BESS,CVE Proyecto Cuarenta y Tres SpA,Matilde Solar,36.00,MWh


,fuente_sitio,indicador,registros
3,EIA,generacion_electrica,8013
2,EIA,emisiones_co2,5622
1,CNE,potencia_neta,282
0,CNE,capacidad_almacenamiento,74


In [7]:
df_export = df_impacto.copy()
df_export["periodo"] = df_export["periodo"].astype("string")

df_export.to_csv(CSV_SALIDA, index=False, encoding="utf-8")
df_export.to_json(JSON_SALIDA, orient="records", force_ascii=False, indent=2)

delete_result = coleccion.delete_many({"grupo": NOMBRE_GRUPO, "tema": TEMA_PROYECTO})
docs = df_export.astype(object).where(pd.notnull(df_export), None).to_dict("records")
insert_result = coleccion.insert_many(docs)

print("Registros eliminados previos:", delete_result.deleted_count)
print("Registros insertados en MongoDB:", len(insert_result.inserted_ids))
print("Archivo CSV guardado en:", CSV_SALIDA)
print("Archivo JSON guardado en:", JSON_SALIDA)


Registros eliminados previos: 0
Registros insertados en MongoDB: 13991
Archivo CSV guardado en: /home/jovyan/work/semanas/Semana 6 Integracion/salidas/impacto_ambiental_energia.csv
Archivo JSON guardado en: /home/jovyan/work/semanas/Semana 6 Integracion/salidas/impacto_ambiental_energia.json


In [8]:
resumen = (
    df_impacto.groupby(["pais", "categoria_energia", "indicador"])
    .agg(
        registros=("valor", "size"),
        valor_promedio=("valor", "mean"),
        valor_maximo=("valor", "max")
    )
    .reset_index()
    .sort_values(["registros", "valor_maximo"], ascending=[False, False])
)

top_co2 = (
    df_impacto[df_impacto["indicador"] == "emisiones_co2"]
    .sort_values("valor", ascending=False)
    .head(10)
)

top_chile = (
    df_impacto[df_impacto["pais"] == "Chile"]
    .sort_values("valor", ascending=False)
    .head(10)
)

display(resumen.head(20))
display(top_co2[["pais", "region", "tecnologia", "actor", "valor", "unidad", "url_origen"]])
display(top_chile[["item", "region", "tecnologia", "indicador", "valor", "unidad"]])


,pais,categoria_energia,indicador,registros,valor_promedio,valor_maximo
11,Estados Unidos,Renovable,generacion_electrica,3857,4.532852e+06,4.519044e+08
5,Estados Unidos,Fosil,emisiones_co2,3641,8.626279e+06,9.193083e+08
6,Estados Unidos,Fosil,generacion_electrica,3264,1.537718e+07,1.869902e+09
10,Estados Unidos,Renovable,emisiones_co2,1514,5.867324e+03,4.701200e+05
9,Estados Unidos,Otra,generacion_electrica,544,4.113105e+05,1.285518e+07
8,Estados Unidos,Otra,emisiones_co2,467,6.012769e+05,1.500151e+07
7,Estados Unidos,Nuclear,generacion_electrica,300,5.197065e+07,7.898789e+08
3,Chile,Renovable,potencia_neta,159,1.870472e+01,2.200000e+02
1,Chile,Almacenamiento,potencia_neta,98,7.745357e+01,4.300000e+02
0,Chile,Almacenamiento,capacidad_almacenamiento,74,3.903655e+02,3.010000e+03


,pais,region,tecnologia,actor,valor,unidad,url_origen
5049,Estados Unidos,US-TOTAL,Coal,Total Electric Power Industry,919308252.0,metric_tons_co2,https://www.eia.gov/electricity/data/state/emi...
5102,Estados Unidos,US-TOTAL,Coal,Total Electric Power Industry,868201231.0,metric_tons_co2,https://www.eia.gov/electricity/data/state/emi...
5218,Estados Unidos,US-TOTAL,Natural Gas,Total Electric Power Industry,820609513.0,metric_tons_co2,https://www.eia.gov/electricity/data/state/emi...
4995,Estados Unidos,US-TOTAL,Coal,Total Electric Power Industry,797825215.0,metric_tons_co2,https://www.eia.gov/electricity/data/state/emi...
5166,Estados Unidos,US-TOTAL,Natural Gas,Total Electric Power Industry,789926512.0,metric_tons_co2,https://www.eia.gov/electricity/data/state/emi...
5113,Estados Unidos,US-TOTAL,Natural Gas,Total Electric Power Industry,742560206.0,metric_tons_co2,https://www.eia.gov/electricity/data/state/emi...
5006,Estados Unidos,US-TOTAL,Natural Gas,Total Electric Power Industry,722489825.0,metric_tons_co2,https://www.eia.gov/electricity/data/state/emi...
5155,Estados Unidos,US-TOTAL,Coal,Total Electric Power Industry,708887031.0,metric_tons_co2,https://www.eia.gov/electricity/data/state/emi...
5060,Estados Unidos,US-TOTAL,Natural Gas,Total Electric Power Industry,695926717.0,metric_tons_co2,https://www.eia.gov/electricity/data/state/emi...
5207,Estados Unidos,US-TOTAL,Coal,Total Electric Power Industry,686354372.0,metric_tons_co2,https://www.eia.gov/electricity/data/state/emi...


,item,region,tecnologia,indicador,valor,unidad
12,BESS Elena Fase I,Región de Antofagasta,BESS,capacidad_almacenamiento,3010.0,MWh
11,BESS Cristales,Región de Antofagasta,BESS,capacidad_almacenamiento,1360.0,MWh
19,Pampas (almacenamiento),Región de Antofagasta,BESS,capacidad_almacenamiento,1360.0,MWh
53,BESS Tarapacá,Región de Tarapacá,BESS,capacidad_almacenamiento,1344.0,MWh
20,BESS DUNE,Región de Antofagasta,BESS,capacidad_almacenamiento,1334.0,MWh
40,BESS Proyecto Algarrobal 200 MW,Región de Atacama,BESS,capacidad_almacenamiento,1300.0,MWh
39,BESS Parque Fotovoltaico Malgarida II,Región de Atacama,BESS,capacidad_almacenamiento,1000.0,MWh
4,BESS Libélula,Región Metropolitana de Santiago,BESS,capacidad_almacenamiento,960.1,MWh
51,BESS Estela,Región de Tarapacá,BESS,capacidad_almacenamiento,935.0,MWh
31,BESS Copiapó Solar,Región de Atacama,BESS,capacidad_almacenamiento,932.0,MWh
